# Birzeit University - Department of Electrical and Computer Engineering
## Assignment 1: Telco Customer Churn Prediction Using Machine Learning
**Summer Semester 2025/2026**

---
### Reproducibility & Pipeline Integrity Notice
* **Random State**: `random_state=42` is consistently enforced across all data splits, resampling, and models.
* **Leakage-Free Architecture**: Preprocessing (imputation, scaling, encoding), feature selection, and resampling are strictly contained within `imblearn.pipeline.Pipeline`.
* **Untouched Test Set**: Stratified 80/20 train/test split where the test set is held out until the final evaluation.
* **Cost Asymmetry**: Analysis incorporates business costs ($500 False Negative loss vs. $50 False Positive cost).



In [ ]:
# Core imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from src.config import (
    DATA_PATH, RANDOM_STATE, TEST_SIZE, CV_SPLITS,
    COST_FN, COST_FP, FIGURES_DIR, TABLES_DIR, MODELS_DIR
)
from src.data_loader import load_raw_data, prepare_features_and_target, get_train_test_split
from src.eda import perform_eda_analysis, generate_eda_plots
from src.preprocessing import build_column_transformer, build_model_pipeline, get_transformed_feature_names
from src.feature_selection import run_feature_selection_analysis
from src.models import get_baseline_models, get_default_models, run_models_benchmark
from src.tuning import run_hyperparameter_tuning
from src.imbalance import run_imbalance_experiments
from src.cost_analysis import run_cost_and_roc_analysis, find_cost_optimal_threshold
from src.leakage_experiment import run_leakage_challenge_experiment
from src.evaluation import assemble_and_evaluate_final_models

print("Dependencies and modular pipelines loaded successfully.")



---
# TASK 1 — Exploratory Data Analysis (EDA) [Core]
Load the Telco Customer Churn dataset and perform a comprehensive exploratory analysis.
- Dataset dimensions, feature types, missing and invalid values.
- Subtle structural issues: 11 whitespace strings (`' '`) in `TotalCharges`.
- Descriptive statistics and class distribution of `Churn`.
- Visualizations of key numerical distributions, categorical relationships, outliers, and correlations.



In [ ]:
df_raw = load_raw_data()
print(f"Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print("First 5 records:")
display(df_raw.head())

eda_results = perform_eda_analysis(df_raw)
print(f"Structural Issue Audit: Found {eda_results['whitespace_total_charges_count']} whitespace entries in TotalCharges.")
print(f"Associated tenure values for whitespace TotalCharges: {eda_results['whitespace_tenure_values']} (all 0 months)")

print("\nTarget Variable Distribution (Churn):")
display(eda_results['target_distribution'])

print("\nNumerical Feature Descriptive Statistics:")
display(eda_results['numerical_stats'])

generate_eda_plots(df_raw)
print("Task 1 plots generated and saved to outputs/figures/")



---
# TASK 2 & 3 — Data Preprocessing & Stratified Train/Test Split [Core]
- Remove identifier `customerID`.
- Handle structural issues: convert `TotalCharges` to numeric representation (`pd.to_numeric(..., errors='coerce')`).
- Design modular scikit-learn `ColumnTransformer`:
  - **Numerical pipeline**: `SimpleImputer(strategy='median')` -> `StandardScaler()`
  - **Categorical pipeline**: `OneHotEncoder(drop='first', handle_unknown='ignore')`
- Split into 80% train and 20% test sets with stratification on `Churn`.
- **Test set isolation**: held out strictly until Tasks 10/12/13.



In [ ]:
X, y = prepare_features_and_target(df_raw)
X_train, X_test, y_train, y_test = get_train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print(f"Full Feature Matrix Shape: {X.shape}, Target Vector Shape: {y.shape}")
print(f"Training Set: X={X_train.shape}, y={y_train.shape} (Churners: {y_train.sum()}, {y_train.mean()*100:.2f}%)")
print(f"Test Set:     X={X_test.shape}, y={y_test.shape} (Churners: {y_test.sum()}, {y_test.mean()*100:.2f}%)")

preprocessor = build_column_transformer()
print("Modular ColumnTransformer constructed (no leakage):")
print(preprocessor)



---
# TASK 4 — Feature Selection [Core]
Apply at least TWO feature selection methods on the training set only:
1. Mutual Information (`mutual_info_classif`)
2. Random Forest Gini Feature Importance (`RandomForestClassifier`)
Compare feature rankings and evaluate cross-validation performance with full features vs. selected subsets.



In [ ]:
importance_df, fs_cv_df = run_feature_selection_analysis(X_train, y_train)

print("Top 15 Informative Features (Mutual Information & Random Forest Importance):")
display(importance_df.head(15))

print("\n5-Fold CV Performance Comparison across Feature Subsets:")
display(fs_cv_df)



---
# TASK 5 — Baseline Models [Core]
Establish baseline classifiers using Stratified 5-Fold Cross-Validation on the training set:
- `DummyClassifier(strategy='most_frequent')`
- `LogisticRegression(max_iter=1000)`



In [ ]:
baselines = get_baseline_models()
df_baselines = run_models_benchmark(baselines, X_train, y_train, "task5_baseline_cv_results.csv")
print("Baseline Cross-Validation Results (Stratified 5-Fold):")
display(df_baselines)



---
# TASK 6 — Classification Models (Default Configurations) [Core]
Train and evaluate 5 primary classifiers using Stratified 5-Fold Cross-Validation:
1. Logistic Regression
2. Support Vector Machine (SVM, with `probability=True`)
3. Random Forest
4. XGBoost
5. LightGBM



In [ ]:
default_models = get_default_models()
df_default_cv = run_models_benchmark(default_models, X_train, y_train, "task6_default_models_cv_results.csv")
print("Default Models 5-Fold CV Benchmark:")
display(df_default_cv)



---
# TASK 7 — Hyperparameter Tuning [Core, scope-limited]
Perform hyperparameter tuning via `RandomizedSearchCV` on the training set only.
- Restricted to 2–3 key hyperparameters per model to maintain computational tractability.
- Evaluate performance gain (Default vs. Tuned).



In [ ]:
best_tuned_pipelines, df_tuning = run_hyperparameter_tuning(X_train, y_train, df_default_cv)
print("Hyperparameter Tuning Results (Default vs Tuned):")
display(df_tuning)



---
# TASK 8 — Class Imbalance Handling [Core]
Investigate 3 strategies for handling the imbalanced churn target:
1. No explicit handling (Default)
2. Cost-sensitive learning / Class weighting (`class_weight='balanced'`, `scale_pos_weight`)
3. SMOTE oversampling implemented strictly within folds via `imblearn.pipeline.Pipeline`



In [ ]:
df_imbalance = run_imbalance_experiments(X_train, y_train)
print("Imbalance Strategies Cross-Validation Comparison:")
display(df_imbalance.pivot(index='Model', columns='Imbalance_Strategy', values=['Recall', 'Precision', 'F1_Score', 'ROC_AUC']))



---
# TASK 9 — Cross-Validation & Metric Selection Discussion [Core]
- Stratified 5-Fold CV ensures balanced class ratios in every fold.
- **Accuracy Fallacy**: Misleading due to the majority class predicting ~73.4% without detecting any churners.
- **ROC-AUC**: Primary threshold-independent metric measuring discrimination ability across all decision cutoffs.
- **Recall & Expected Business Cost**: The primary operational metrics in telecom churn management.



---
# TASK 10 — ROC-AUC, Confusion Matrix & Business Cost Analysis [Core]
- Select Top 3 models based on cross-validated ROC-AUC.
- Generate single composite ROC curve figure.
- Generate confusion matrices (TP, TN, FP, FN).
- **Financial Cost Optimization**:
  - $FN = \$500$ (missed churner, revenue loss)
  - $FP = \$50$ (unnecessary retention incentive)
- Optimize classification threshold: $t^* = \arg\min_t [FN(t) \times 500 + FP(t) \times 50]$.



In [ ]:
top_3_names = df_tuning.sort_values('Tuned_ROC_AUC', ascending=False)['Model'].head(3).tolist()
top_3_dict = {name: best_tuned_pipelines[name] for name in top_3_names}
print("Top 3 Models Selected for ROC & Cost Analysis:", top_3_names)

df_matrices, cost_curves = run_cost_and_roc_analysis(top_3_dict, X_train, y_train)
print("Confusion Matrix & Business Cost Summary:")
display(df_matrices)



---
# TASK 11 — Data Leakage Challenge [Core]
Compare:
- **Experiment A (Correct)**: Preprocessing and SMOTE re-fitted strictly inside each fold.
- **Experiment B (Intentionally Leaked)**: SMOTE applied to the entire training set before CV splitting.
Demonstrates the illusion of cross-validation inflation and its failure on unseen test data.



In [ ]:
df_leakage = run_leakage_challenge_experiment(X_train, y_train, X_test, y_test)
print("Data Leakage Challenge Results:")
display(df_leakage)



---
# TASK 12 — Final Model Comparison (CV vs. Untouched Test Set) [Core]
Compile the Single Best Pipeline for each of the 5 models incorporating the Task 8 justified imbalance handling.
Rank models by `Test_Expected_Cost_Optimal` ascending under the asymmetric business cost regime ($500 FN vs. $50 FP).



In [ ]:
df_final = assemble_and_evaluate_final_models(best_tuned_pipelines, X_train, y_train, X_test, y_test)
print("Master Model Comparison Table (Ranked by Minimum Expected Cost):")
display(df_final[[
    "Model", "Imbalance_Strategy", "Resampling_SMOTE",
    "CV_Recall_Mean", "CV_ROC_AUC_Mean",
    "Test_Accuracy", "Test_Recall", "Test_Precision", "Test_F1", "Test_ROC_AUC",
    "Test_Expected_Cost_Default", "Best_Threshold", "Test_Expected_Cost_Optimal"
]])



---
# TASK 13 — Deployment Recommendation [Core]
Select ONE model for real-world telecom deployment based on:
- Expected business cost minimization ($500 FN vs. $50 FP)
- Discrimination ability (ROC-AUC)
- Robustness and minimal generalization drop
- Computational complexity and inference latency



In [ ]:
top_model = df_final.iloc[0]
alt_model = df_final.iloc[1]

opt_cost = top_model["Test_Expected_Cost_Optimal"]
def_cost = top_model["Test_Expected_Cost_Default"]
cost_savings = def_cost - opt_cost
pct_savings = (cost_savings / def_cost) * 100 if def_cost > 0 else 0.0

alt_opt_cost = alt_model["Test_Expected_Cost_Optimal"]
cost_diff_alt = alt_opt_cost - opt_cost

print(f"*** RECOMMENDED DEPLOYMENT MODEL: {top_model['Model']} ***")
print(f" - Imbalance Strategy:       {top_model['Imbalance_Strategy'].upper()} (Resampling_SMOTE={top_model['Resampling_SMOTE']})")
print(f" - Test Expected Cost (Opt): ${opt_cost:,.0f} at threshold t* = {top_model['Best_Threshold']:.2f}")
print(f" - Test Expected Cost (Def): ${def_cost:,.0f} (Savings: ${cost_savings:,.0f} or {pct_savings:.1f}%)")
print(f" - Cost Advantage vs {alt_model['Model']}: Saves an additional ${cost_diff_alt:,.0f}")
print(f" - Test ROC-AUC:             {top_model['Test_ROC_AUC']:.4f}")
print(f" - Test Recall:              {top_model['Test_Recall']:.4f}")
print(f" - Test F1-Score:            {top_model['Test_F1']:.4f}")
print(f" - Serialized Artifact:      outputs/models/best_churn_pipeline.joblib")

